# Fine-tuning de Qwen2.5 para un tutor de matemáticas

### Notebook 1 de 4 · Arquitectura decoder-only (generación)

**Proyecto:** Tutor inteligente de matemáticas · Comparación de arquitecturas
mediante fine-tuning
**Curso:** SI4006 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Notebook base:** `S04_Lab_Fine_tuning_Qwen.ipynb`

---

## 1 · Introducción

### Qué problema resolvemos

Un estudiante escribe una pregunta de matemáticas en lenguaje natural
("María tiene 48 caramelos y quiere repartirlos entre 6 amigos…") o una
operación directa ("(45 - 9) ÷ 6 + 11"). El sistema debe devolver **el
procedimiento paso a paso y la respuesta**, no solo el número: un tutor que
solo da resultados no enseña.

### Qué arquitectura usamos aquí

**Qwen2.5-1.5B-Instruct**, un modelo *decoder-only* de 1.5 mil millones de
parámetros. Es la arquitectura de los modelos tipo GPT: procesa el texto de
izquierda a derecha y predice el siguiente token, condicionado únicamente por
lo anterior (*atención causal*).

### Por qué este modelo y no otro

| Criterio | Justificación |
|---|---|
| **Es generativo** | La tarea exige producir texto libre (una explicación). Un encoder como BERT no puede hacerlo: solo produce representaciones o etiquetas. |
| **Tamaño** | 1.5B entra en una GPU T4 gratuita con LoRA y entrena en minutos. Un modelo de 7B necesitaría cuantización agresiva y mucho más tiempo. |
| **Multilingüe real** | Qwen2.5 fue preentrenado con volumen significativo de español, algo poco común en modelos pequeños abiertos. Nuestro corpus es 100% español. |
| **Variante `Instruct`** | Ya fue alineada para seguir instrucciones, así que el *baseline* es un punto de comparación honesto. Con un modelo base puro, casi cualquier fine-tuning parecería un éxito espectacular. |

> **Alternativa considerada:** `Qwen/Qwen2.5-Math-1.5B`, especializado en
> matemáticas. Se descartó como opción por defecto porque su razonamiento en
> cadena está optimizado para inglés y chino, y produce explicaciones
> inconsistentes en español. Queda disponible cambiando una línea en la
> celda de configuración, por si quieren correr el experimento con ambos.

### Ventajas y limitaciones de la arquitectura decoder-only

**Ventajas**
- Genera texto de longitud arbitraria: procedimientos, explicaciones, ejemplos.
- Un solo modelo cubre todos los tipos de problema; no hay que decidir
  previamente de qué tema es la pregunta.
- Se adapta bien con pocos datos cuando la tarea es de *formato* y *estilo*.

**Limitaciones**
- No tiene garantía de corrección aritmética: el cálculo emerge de patrones
  estadísticos, no de un motor de cálculo. Puede escribir un procedimiento
  impecable y equivocarse en la última multiplicación.
- Es el modelo más caro de los cuatro en inferencia (genera token a token).
- Su salida es difícil de validar automáticamente: por eso invertimos en un
  formato de respuesta estructurado.

## 2 · Objetivos

1. Medir el desempeño del modelo **antes** de entrenarlo (baseline), sobre la
   partición de validación y con métricas definidas de antemano.
2. Aplicar fine-tuning eficiente con **LoRA** sobre 132 ejemplos de
   entrenamiento, con QLoRA disponible como variante.
3. Evaluar el modelo entrenado **con las mismas métricas y los mismos datos**,
   y cuantificar la diferencia.
4. Distinguir explícitamente dos efectos que el fine-tuning con pocos datos
   mezcla: cuánto mejoró el **formato** de la respuesta y cuánto mejoró la
   **corrección matemática**.
5. Dejar registrado en W&B y en `resultados/qwen.json` todo lo necesario para
   la comparación final entre las cuatro arquitecturas.

## 0 · Preparación del entorno

Instalamos el ecosistema Hugging Face. Las versiones se fijan por rango mayor
para evitar que un cambio de API rompa el notebook meses después.

> **Antes de empezar:** activen la GPU en `Entorno de ejecución → Cambiar tipo
> de entorno de ejecución → T4 GPU`. Sin GPU el entrenamiento es inviable.

**Sobre la desinstalación de `torchao`.** Colab trae `torchao` preinstalado, y
`peft` comprueba su versión con una función que **lanza `ImportError` en lugar
de devolver `False`** cuando la encuentra más antigua de lo que espera. El
resultado es que `get_peft_model()` falla con un error que no tiene ninguna
relación aparente con LoRA.

Ningún notebook del proyecto usa cuantización de torchao, así que lo quitamos.
La alternativa —actualizarlo— también funcionaría, pero torchao está acoplado a
la versión de torch y actualizarlo puede arrastrar un torch distinto y romper
otras cosas en Colab. Desinstalarlo no afecta a nada de lo que hacemos aquí.

In [ ]:
%pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" \
    "accelerate>=0.33" "bitsandbytes>=0.43" "scikit-learn>=1.3" wandb

# Ver la nota de arriba: evita que get_peft_model() falle con un ImportError
# de torchao que nada tiene que ver con LoRA.
%pip uninstall -y -q torchao

print("Librerías instaladas. Si Colab pide reiniciar la sesión, reinícienla y sigan desde aquí.")

In [ ]:
import os, random, sys
import numpy as np
import torch
import transformers

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
transformers.set_seed(SEMILLA)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers  {transformers.__version__}")
print(f"torch         {torch.__version__}")
print(f"Dispositivo   {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU           {torch.cuda.get_device_name(0)}")
else:
    print("AVISO: sin GPU el fine-tuning tardará horas. Activen el runtime T4.")

# Detección temprana del conflicto peft/torchao. Más vale que salte aquí, en la
# celda de entorno, que dentro de get_peft_model() veinte celdas más adelante
# con un mensaje que no menciona LoRA por ninguna parte.
try:
    from peft.import_utils import is_torchao_available
except Exception:
    pass                       # ruta interna de peft cambiada: no es un problema
else:
    try:
        is_torchao_available()
    except ImportError as e:
        print(f"\nAVISO peft/torchao: {e}")
        print("  Solución: ejecuten  %pip uninstall -y torchao  y reinicien la sesión.")

### Persistencia entre notebooks

Los notebooks 3 y 5 **leen carpetas que producen los notebooks 1, 2 y 4**:

```
1 · Qwen    -> adaptadores/qwen-lora/      ─┐
2 · BERT    -> modelos/bert-clasificador/  ─┤-> el notebook 3 las lee
4 · FLAN-T5 -> adaptadores/flan-t5-lora/    │
1,2,3,4     -> resultados/*.json           ─┴-> el notebook 5 los lee
```

En Colab, `/content` se borra al desconectar el runtime, así que ese trabajo se
perdería entre sesiones. Montando Google Drive y trabajando desde una carpeta
suya, los artefactos sobreviven y cada notebook se puede ejecutar el día que se
pueda.

Con `USAR_DRIVE = False` todo queda en `/content`, lo cual es válido si
ejecutan los notebooks 1, 2 y 3 seguidos sin desconectar.

Fuera de Colab la celda no hace nada: el directorio de trabajo se queda como
está.

> **Los checkpoints intermedios nunca van a Drive.** El `Trainer` guarda en
> `output_dir` el modelo *más el estado del optimizador* en cada época. Para el
> notebook 2, que hace fine-tuning completo de BETO, eso son varios GB que
> además se escribirían por red. Como son desechables —lo que importa es el
> modelo final—, se mandan siempre al disco local del runtime mediante
> `DIR_CHECKPOINTS`.

In [ ]:
import os
from pathlib import Path

USAR_DRIVE    = True
CARPETA_DRIVE = "/content/drive/MyDrive/ProyectoIA"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_DRIVE)

# Checkpoints del Trainer: grandes y desechables -> siempre en disco local.
DIR_CHECKPOINTS = "/content/salidas" if EN_COLAB else "salidas"

print(f"En Colab           : {EN_COLAB}")
print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Checkpoints en     : {DIR_CHECKPOINTS}")

## 3 · Arquitectura del modelo

```
                    ENTRADA (prompt tokenizado)
                              |
                    +---------v----------+
                    |  Embeddings + RoPE |
                    +---------v----------+
                              |
                   +----------v-----------+
                   |  28 bloques Qwen2:   |
                   |   - Atención causal  |   <- cada token solo ve
                   |     (GQA, 12 heads)  |      los anteriores
                   |   - MLP SwiGLU       |
                   |   - RMSNorm          |
                   +----------v-----------+
                              |
                    +---------v----------+
                    |  Cabeza de lenguaje |  -> distribución sobre 151k tokens
                    +---------------------+
                              |
                  se muestrea 1 token, se reinyecta, y se repite
```

Lo esencial para entender el fine-tuning: **la máscara causal**. Cada posición
solo puede atender a las posiciones anteriores. Eso es lo que permite generar
texto, y también lo que impide que el modelo "lea la respuesta" durante el
entrenamiento.

Dónde entra LoRA: en las matrices de proyección de la atención
(`q_proj`, `k_proj`, `v_proj`, `o_proj`). En lugar de actualizar una matriz
`W` de tamaño `d×d`, se congela `W` y se aprende `ΔW = B·A`, donde `A` es
`r×d` y `B` es `d×r` con `r` muy pequeño (16 en este notebook). Se entrena
menos del 1% de los parámetros.

In [ ]:
MODELO_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# Alternativa especializada en matemáticas (razonamiento en inglés/chino):
# MODELO_ID = "Qwen/Qwen2.5-Math-1.5B"

USAR_QLORA = False   # True = cargar el modelo en 4 bits (menos memoria, algo más lento)

DIR_ADAPTADOR = "adaptadores/qwen-lora"
LONGITUD_MAX  = 384       # se justifica en la sección 6 con datos reales
MAX_TOKENS_GEN = 200      # techo de generación durante la evaluación

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USAR_QLORA:
    from transformers import BitsAndBytesConfig
    cuantizacion = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",              # cuantización de 4 bits con distribución normal
        bnb_4bit_compute_dtype=torch.float16,   # los cálculos siguen en fp16
        bnb_4bit_use_double_quant=True,         # cuantiza también las constantes de cuantización
    )
    modelo = AutoModelForCausalLM.from_pretrained(
        MODELO_ID, quantization_config=cuantizacion, device_map="auto"
    )
else:
    modelo = AutoModelForCausalLM.from_pretrained(
        MODELO_ID, torch_dtype=torch.float16
    ).to(DEVICE)

n_par = sum(p.numel() for p in modelo.parameters())
print(f"Modelo: {MODELO_ID}")
print(f"Parámetros: {n_par/1e6:.1f} M")
print(f"Capas ocultas: {modelo.config.num_hidden_layers} | "
      f"dimensión: {modelo.config.hidden_size} | "
      f"vocabulario: {modelo.config.vocab_size}")

## 4 · Carga del dataset

### 4.1 · Materialización del corpus

El corpus vive en `data/math_tutor_dataset.jsonl`, generado por
`scripts/dataset_fuente.py`. Para que el notebook funcione en Colab sin subir
archivos, abajo va una **copia comprimida** de ese mismo archivo.

La celda **no sobrescribe** el JSONL si ya existe: eso permite escalar el
dataset (reemplazar el archivo por uno mayor) sin tocar el notebook. El hash
SHA-256 que se imprime debe ser idéntico en los cinco notebooks; si difiere,
alguno está entrenando con datos distintos y la comparación no sería válida.

Hash esperado de la versión embebida: `cf4732834d64a196…`

In [ ]:
import base64, gzip, hashlib, json
from pathlib import Path

RUTA_DATOS = Path("data/math_tutor_dataset.jsonl")
SHA_ESPERADO = "cf4732834d64a196d49eda88ddac8694a529a8bfc6b843e8eb8d74b8c6d59ecc"

_BLOB = (
    "H4sIAAAAAAAC/9V9XY/jSJLYuwH/B2KABrrR3ZL4KaqAQWM8O4DvMLc3t+Pzi20sWBK7mmNJrKWkQlUfDPje7h8s4Ld5nId5"
    "2BsbBublgKkf4P+wv8QRkUkyM5lBJlWqKhWw21OimBmhjMiMz4z4py+K1RcX3he7w+b9zP/iHfx1vS72+GhfZcUWnyyzfX5V"
    "VkUmX8y0h3+kCWbwaF9cl/hKeZ1X2bIoafC2uMnX+PQy2xXLEh/BqKt8K5/l+CTfArAVzf+HfHfI1ze5t868XXF1KOC73JNT"
    "3v+yvfCCKPbeeuE8nRC62boQI7/LdqXnX3jfA4abcucdtvDFKt+981b5Mt9mO+/OW+Js+CeM2ubZdlUCnJ23LD5W2W7yX7c0"
    "R6DA8L70kiCEbwCv60O+22fex2KbrS/wMcK/hhE7AP9fvjgeLs5jQvziv8HDqgb6RwKKv1KCvcnWZaV8znd/XOUbXP6P2XqX"
    "/49//+/+SaVs8CIoG07muAjBJOJo+xUtH64yTLIsNxmucbHJ1l4Gk+3gk4c/o1JIqUyKSzthqDkxyekMCodZgLAEnJgUnDiR"
    "MDwNCa+r8nKdi9fGUvBvD9nWq/JlWVVAMFjZ2Pvvxfr+l02+r2CdrssK12qT3f8l22bA9cEktb2wz6pVPvF++7evD/c/bvfw"
    "hfpSO3++9fblPlt/sPPCN2vxtZcTifC3I1FwX61gnlUBy75dFtq+RpTfEl5IJYYTQhUfnSncYQqu0KHxLNEBqXCHC3NET72/"
    "t4f8phy9wf0gnuEpF6WzgcN7XYoV3d7/uslhTRQaKpPAukbzcGYjIz63n8+dqfE1y6QcteqZGxLVDwZoFL8MGs3EUTabJAyJ"
    "vtktq+KywKWEl/BoLPGPmXedVZkHc2brrBKrLA7MXKVeOz0utD8JYxv18Lmx846AijNY4HGErYE2hK0fDBA2eSmbDw8j0jJS"
    "x913XRW0RdrBX4KKolCzHpDDuZhXS3gXztg5gYBX51bazuP+fckAxUGD4DjSznXCzl3IOn9igWuj6j+CHL0sLtdFuc+XIDnz"
    "7f3PmeeDrrguLlFa3qG8hH0B4jJd1A9ppp0mYeU3e2CL3Ms+lVX2gdOw5IyVPhutOYm+7LABDLN3Xra7/9n700HoQRv9jCZt"
    "FjD60gvCyMYF8FgCMNWu4+HTQa5D5hhCB98wBjx24IxU44wbWsO93LhPyh5fZ5sC9jvSswIO8OPZDA7EfCeWa33Y5sghAcg2"
    "5THorKC/KtzRjB9WuOjN0q4Dye27Ag7VWIGQeiuxwK0M/7VxBH0h0OwoXc5wiQWsEDlOMMA2rEDPHZhhcS7M8BVYnh7+tlVR"
    "evS8Kreen4Jik60O6z2dFsDgM29b3P9FPR9AewU5sSvRQG0GDjDD3x5gnHpsX1WHa5K+zVwqE6SkXhF0UK/8hV1ng+fNaJ0H"
    "XMAR7buAWD3OhNYqdPDNMOX92WPK/QLEd7XJgZankP3RbJHAsiyC9Hi9u50DVjaeBdZjHZ+PULu7c3LUqiduiFQ/GCCSfyZS"
    "vDrs6bja5dtVXhU7UF+FMIZNKzQeD96Hp/BOMAl0y/oO39rlVwd0HDWvkdNBea3e0PUJuS63V8X+sJLHJtmphMeHAR5QhioH"
    "LMHVnGQToU0iGmjg2oxp3pR2g0auMQMOa0bzRrQDowQvZzf76WQBCzKfzOJhG41eJnMJ/3K00sSrAgRqUvFkYdXl6QvWUBsB"
    "WpzdFqCsAldDbnW3+skAocMzORFw+68y7wb+RfU9mntLUOd2OYpjUNN29z9eZrDd77wkxM+rclNsr0pNarcD6llWhwrdkzgA"
    "lkocOBvwxg1ueZoCZ+pR5SI0tQAbsKN9qwCHxy1O3HbnIZFE0GCwproJqLXZfRfRHZ1QaXuKHT9PyawJ/Dg+XoArk+DygpJs"
    "JyJ+MUKG26Zl6VbP3dKrfjJAsfgMdu03W5DDYJCXq/wq8z5ldyClctyFPwhuho32OUMleomKdwBwFzHu31kTE9J2b/M6zjSg"
    "bqvbp3WSZBI4ukrw5FzQuenHFg8N+t3JZZLRkAt8DXcaqsmB3VEDjxsc+c3sgI3quhnCgz3vDWTaY3/ImQOTuYU2K4Ro4R//"
    "6SJgKbi83ntRvGB2+R8QRVxGWxwRWO26WKFUKYEiZAmCnliCvgkcBgwLC7PPFNZogKGNHFuDIfBYp/wJEMAJTdCscR7rsTL5"
    "eYDYwQsh9iSCRQgnwXxYi8N3SZOCPxx1OHpTzI/G1cS3UhifswqcM1SiaRcea8xJoK0xJx8MEDZ0Fdsu1LWKAk5sWwj8LZ5k"
    "G6TnHpQuD080YPQEPBMZnolf5+tdcbBERUEMLKW2tqhfrtU2eHFbLj/lrSn38ZCLM7OFRMvIyIl/3Mr54eMuB1cq7kNUtzxa"
    "EtiZCockQC7cge/Dd14uyFm9EzJN4nWZ/VB66Ge3sc770PitOh85oIIDxiDB8ZMVk4a53ruwVvTkZ8ZxmiD5Nt+DhJ3HQzKi"
    "RxVUZiEHaRDbPbJBzBz/Vk3QMivvdQ1iw90auMjx+IWQyaeVgLCVi4UO7worGTRnVwOdXhUQKGNoEtuTlCY95rkzXLLOLRD5"
    "JKVJbGQpTVyom5yIuieNuZGXLgSNtgm5Cd0Gg1y2GBtEpFaQJwO5INflFubJd8xZ/W07ptaWdnhS4ql5wIMSD1AZ9VJzWALa"
    "ZBFZW3bNHR5bg2ujIFIGiwGLtes0gK1x56Sbz8+D6pCXtMVoIhhvguaYEgJIVSLMkMHOANrDah32xboAQ8RLU/m9wQb7lg0+"
    "DJzRsNo0sZwT/EHgqUUMcrAYS6/YFpBNpFIfkXqPkIEgsKtsxE8CiQNzdjvCJNerDo0lvwayJX8SOJA/fdQj/cSe15kQcUn4"
    "IPGrToOSMmHkbzJO/tqmZQVwYgrgxEkAL85hs361XVX3P+3qfIhIiXvfeVfZbg+6dZA2cV011r0CsxUiyetcbFA+CwKngfdK"
    "oa1u6BBY18O7+zKSyo8Ai9uFCXPTF7YwtytM8pJaofHb0x7ipufDNDcCnQ80u57EXTpB98JskjpoX/guaUHwh6PyRW+K+UVm"
    "Z8wkkvK6lzNU8rB24fWkksZm/qjDtvb9x9zWY4j7ffkRt/SmAJHrT5LUa0Kgu4P3CabJtmX9bRx49sjnqviYVzkm4qLkRrSF"
    "iQ5gMDQCK8ypZNpYkPXlpdAFapMVZ2/mgzwkMO4zYd7flWrWOaH+XuD4JVDOt4ZH8blnC44+EA8SBl0MOKYx0Gh4B5878E7w"
    "QmyyZIb+NvbeiIvgrqdAB1dgJSk8HiOyjQlZh5mYtfWXBS6EeXx32RjtOs/2VZ30gPIIhuZbkZUExj9yM0hor1wertEs0VTq"
    "5lVpW6GpwZpV2oI3I+t5Udc1LapYGuu4UXxIkbFJbchQqefqoW8POOkg0QCxAtuA1krsIHWge/SCNOr3MXmMmD35HeQqFkIP"
    "wjcx2xWYaJnjr99hfnAdWMP4GWa35Eug2fawuf+pKpZqsEPCQUejP7d6M/25TtiHwsbZdKis+1KAbh2W4vMAleNzEdkQFv0P"
    "5VUJ29f0VItrUn5k+sQp3Ama7e/zGwoRwZ9/OBSf5fsdb26/eFcAOkn1fGeiSUKTpCh9ReJUS2YGIr5+H77BXRthJhJ5Q+yO"
    "l16X+FGIiKxmOwq8P6bPIT7EXpvD2i1iCi8CF62B25emWBF8Fjzh5cHEu/+zhxmEVh74Hbi4yg145HD3tm+TmUx/0vUf+jNS"
    "VX167kf4YppYUx7hsU7lUaBIvzeAcFSVkBo6ys8DhAyehpAjImcOqhr4mGF5/AknGP6uQRgPY7KkdgXEzrxPh8uCkgvgxiYs"
    "fiLmwRN4ns4Uwn6D9wx2MAseQEIpwe0iA9mN/YXGHl0Jg/9YL7NMUp34RyKGkxyDEnvhReDV3niZpA6sEp6aVR4aYwW9MTvs"
    "y839LzfF2rspIGME7BxMwoG5yiXKX/Rcok20Rzc27NfOjVO87DJw2zSvQHaBSAroZdJIZ2BkbgE0p2V+XW5vcqkgoMu8gGMc"
    "blKICS6sM2GGC/A1faFw4u/qS6PwffurgDnknF/ij8Jjg+Iufmr3KgFHsKm2D8UVZxuPJSubuqi2kimdObCpc/bew3nVUTx9"
    "l68qeY8mw+ynxh25RPe6SMG0eCE/ZZfIeWIcKUKwWujYBE5O5TDezLnO93BLCm/0UDq5vCNzk8GvwxzLJWqmkGXWrIGm0xCO"
    "QC+0PBcJkwpIX9h8lccAF9dsO2A5NjFgNyxCzx2YJH4+/eXoy9G0Mk4Sr/9+dL3Cfk+KZ5/Y4q5Ia/P25Xh2UjwdCJa8NIJB"
    "qjKsRzhzIph8mfR3H45aTAdaFaJeSJ1ho+RZ08TiZTsFI7+XgsPgZLK1BoglqYTWkjTyXUg6fxJ9YoQTKltvsuX9T1txxxRM"
    "01hm796Jkxo/4NWTHPX19f2P18UyN6K78mFz7/aDC/XhnJR7CY92AZISw9QZ6ZsVqTfa/UnFfLBfnITk2XqSPp4YjYS4TKmB"
    "58OJOg5KWNGFUdJHkehP4b8KhKUSTSKnY2AHlzKkBSCOUzL6fH92lG3i+xMR/vPttzH6uKEHk1HGiIIDf1vDuKLhwBGL59Xx"
    "bMzwrYggUcIEHqObwy6HxESx5m20mfaUvFeq6nzXkAK9hZzL+sqp68FxjSdNSbPqO9hyz5awAFomGE/gpD99YdPqjgBNib8d"
    "oGwYw7frdLGbiuB04/a8dLqZNPPdvFLN66Rjib/JNST+DrQUEnHR2ae9F9glAz7v9U0NAhRpJSYoPm6RGHpC4HL8+/7Z6QmH"
    "VbGHt+qjbw4rum7VBPwgv/HBTivW8J12tUc8crnY0yeiBUyRQN7MWQto/FJ1VBLtUkEhhhfkFO46wjAC5L7UQPfwhgJf5RAH"
    "BnlmH+axlZSEYE041YBeyP90KGA1ciFTlgeIOpVKbZNVcVMIxT2JZAJDpJVTqmFQFpA94Ugn+NFQRTklHR6fcmRkGjkQ+Xm9"
    "j8xJcHlAR8ayxMwc6SwEKav5EGk34A3WIS8jngUQvaHLrq7HgSyHh86hIquswti8ppuKc1yocwvmou6ix194PBJCHdDB81rh"
    "oscROFiCg1gmeonnAtYmQeEaO4aq6tfRBJN/i1p19Ld62TNJqfCNPyeNPGUyENN41q8VDEKkzJUuLDZuFRseoTR2UvfiMzwQ"
    "tHv7OdxcuizXOyEVM/CAfhb1O+iAKK8qEQSq7WnjeKi/l3f3j1AQJGy5G1GvZpwIi9Y1HzBXRQIda3clYQAJnMgEz17y7eLQ"
    "3vMdvEICQsstbI3SbWfnlvBEAYG/yypMYxQ6IhSFBQd4tsnXlO0EohdSAPBCHGaYVGtpMlL+pxS2iQfFDa6MLKh2DlkhDUIH"
    "tMz0Lh8mICgoe66p8pcAJPzzQs6rgQHA9f7/kilnjXcqv6QTEXCAQ05HBQJ7Yuhg2oPDgQOC03HAY18ABi0KloK3DhstrMfd"
    "30xCIThrjDowTnt+WpLf+oRshDnQA8yBA2nCxybNSZMQoHJBgisxc/TtZZtLuZV9mdINOx2KHVP1Flk7WuTzwy8sKwRA80ea"
    "t1c8Qsm7sDp7F30n8/EoCF+vCpx19S50B+/CgfDRI5/K41IKQJvPr0CLLSvwyMENKEooyG+Xh2qHlBcLB4XJ4HeDH4DKfU+8"
    "r/GkRWtAnOnL7DoTMXB6GxI2tbeVYxvG5HTLbotR0QIv2dEQwGgLVwQrJA98KldsrkGzYTEiixSicmnhJAq9DJbjFsi6yrAo"
    "Zq5w0u/p1jUQD4TxYQcw8EeC83ZJ3E2C+7BTTL4qp0oNxEafMgz0gz5TXEKFjkj8BGvFPvEVc74Mo4sDT48oW/Gvxbat9efA"
    "vfF56BQuSQaivkCO2aiQnlqJ1zrXoNCZI5ILGqb8MCiCMB9x316iqEP+YqK1mVswEzJkJmIDTBS6xoKVTwMwm5QCExofm1ZB"
    "KhFqBy5InkOvOP6aIm28+CGaRTsLltK0V+ycjVItzBnZjWoUWHchz9w1YHTqncpWimrkDLqBUbYqEkLULa6NgJzKrUZ1UVOp"
    "OGtSxBiLtyBgCO18GjW4f5v6yc1mUmw4AXicHqKixHGBC9AxmocJc5wWkj6X+nnsLcaYNgtbQ4LX/bDKujwAlBBB3Ow9e7En"
    "V81SmZ38/+q8LO10cjlQa3FOG/p7YN4NaHyw7QLwgl1l67Uo1AgeXQjpZrAlFZNXk7TNu/Cfm0xsWTngYZu2hqr1LRKSEK1r"
    "314aIm4QOmrTNkBF0yINHCt1dZit2DUcOvvqYHKBU8j3TKTuIqKNMH+IzK3nwAVlWmOYt4oHZK4xI0sjs/dF6OBt85/D28b5"
    "ZT9iMXywBuT9/zCeKdXwtx6UDwTFGAwJVCohZxV3IhT6oAxy3JIbvSa+nOWa7uPsXDaquLzP7xwCr9ZxiWvFFU/OOVeOcz5j"
    "sjWOAE4lXSxgWR/PzJ6vMR8u2knsEbwsgeuLdUkng2Y4+icxfTCoU523NBguqZeXYNZrLh0tkyq/zZZAqBwvENr98H6nZNMo"
    "uJ3cLQMiH543Sjb5Tv529zvGj+HeYSS1cLCjgnyToUMhjZQ6Pj8crrA9AoQt9nmdz4CNusxKPvBN8woo6iC+tmw5H20r3pRr"
    "7FCibUZsIdc4jOroiJxe9eWS5zUUgRJ725SowarnNBiNAvl9deBsjEbHoI3POHhS/OhlHQhRmtCJsGDblRUb+DEfpY5MvlYs"
    "ziBzrJHNSG+mmWgiLdIiHmHWnL2XgtlJYTQwEW5RwbCZeUZzBRdixs/k1GUyNWCOjyCpqjq+9rGEYiEVZbrITiG6BY5Fmin3"
    "pRIRyzhkXbj1HBgkBzVWFE2j+bfDLluYt453EUxY/WJ1KLXA/fflmjJJqQTb2gZRbXcEk6TNG3e1OyDWfh2m94pX7GE8OZyV"
    "NDzWVM35MfHlg4IK0q4hQUDILSjcnCcwFwy6BKgrccQZfBs9XYDQp8YX6RyLRvSUhC5hElmooPhM3E8pytBnJF9+ypRpsBy3"
    "mkRa167AKtwBAaFr5aH9anu3FPRIwDieBclfZQ+Ny+uhA8WDp6X4SeOOGCD/s0dl0cGlPWFrGqipLxgpEYEpqMUO8EHIKxNh"
    "IHfWLQrfXG8PKIXH7uJNTR/vEZCVAvBdmOx2N7zA6ZAbmAjvrI+e435/HRLR0zdtbYDeDS9bJVLeDSSc/IQnaYEJXGIaFPmh"
    "QnbNoyeO+D+Tz1WcCl1l0w/YLe8EGkf3AmWVTF/PLpCfB2gfvdBjfiG0TCzXxUZsqHQI6lLW/dbM8CUdGbYT3qcacuSltffy"
    "MRv5jIKoH+06LPaGr25wDjXvIRLHL1WSR8j2r0nkxW+O39liAir+wO5rAQuF7sKaw7VIR+9qBaxlT2sAWWrrBSkWLnpb8pIP"
    "c1Hsy5+LfqTHa2/tPBBfCWKbGA+Dpulp5Ef21orRaO3NBKxJcQMk32QxMlorRg50n79g7e019nsHETiZkxQPhwlvbrZmAlzb"
    "Scxv9EjcPaVb3iHjTQwnMUv4IcCWrW6C5MMJpjsxnLic7+kLpryPq5eIo7Cnpp1Ff4awoDx5L+oJgAKGpTZobCH4CLM2k5no"
    "OiVLz0VWKRClg2o9h5YU9g9BiD0vdCkRuUiJxTNIidMeGRGWCVy8ER6Xt3SVeuyhIaagW/pabaHGoRMmtUOneSFURIiA21PY"
    "zvkUUTERlYOsOGjSRIXOOwJGFbFD1nCKHp8rV5Ch9Br3UPyGmjCMZwoaLI5qVogQGD+sE31qxmiVefDQvBfXuQKmB6gzayj4"
    "WGSLholuUGg48C1BjX6gDgziv2RNQyTqC2/Bax+33dzBtJAJKApZ5Dzk68VMNZxIXgBhmCatKwHZSyImvLI5ArqFRVS4bAl0"
    "XeNMHBRO/9HchQ/J95XF7y7vf90pyeJ4iMLKZj9Qhof3DdWelQ0YKbEcw3pQ2wV97k1Xd6yefolJEFpYo5lGySjEJFsJ1KW+"
    "Mvr3CRod7u+JNkFq6z8pXyY8LjAlXBwFoT0VLWyR6ym3zEDX+k3ycNlUNQN4m7Xm4Hl2usX8DKwkaxu1zW3U+kbfF1pfm0Vb"
    "DujOWx8AIfw6u6pyrGACKhxF4CDE89u/LTt1kTDZFdO0M1kWGcG6cFIzAdwBWFF8IJJ6hL2fKeY5ERoXtaucOojaG5ky9ZHc"
    "ETBZygqa711qr40UuKgw0ZPy08i6z1gFB7q5Yp7JLqfgH2ahhFhQAMyArTAOoli/qXDYYlfroO5EhYHZWecNNRcLr4dkV5CN"
    "NVBL4+saqtQlItkcJYy18krfElyse4mFDOSdgWDWvhN22ylTftXuQswlwhc0LoiZS9b0hS1tqxdJfJVHz2TCYcR4VcneEYae"
    "OzBlfB5q05F51QEqHK8joVkf56OJpBhZ2I0tAWKhKlGhXYkKe1J93E2uBhvD4FLx4NSpcCjfZ1Saz3W5d4u3w4tUBn33R/ha"
    "nowGY8RP6Kz97V+xwMFv/8qwwtfZenlYN02vG+TFSNxuWD6X/lzYPLTkLSWj2Err0KD1KHCaJa0CYhUcnaShC0mDF0fS4Ldf"
    "+sKo2hI3KyyGDds+dFPFGjQNgx5SWuBw21JCYGmoB0jDwIGG4Yuj4V//5X9hWfy3bE4ueC2r7P7nz6RxkO1DDScike+K6duo"
    "/PtBW+Usimz7E757K8wHuzTv5kCMB6vtUxUge/TqCRDBzIHG0UukcRCg1Q9/RAu38xcXf4nVJ+Vg1NHhRKQZjOCoEpVWHRvd"
    "PJfeE3gQoBEAV7wYXHrLmCw2Im38VKQ9rZdqAWIKkwHeiGsob6Xv1/1EXpCcS317OKTjY/Tr1INYdXS3+dHoC5GbUNw6nEvX"
    "kwjf4F61V1x2O9cbbI0oSS+emtrmhCFfqtko1OzAWMlznhlHZmb/9gvX2hFv2VXU0vZwWRpF6NVbMdRinq5RoAJNEtdo/hLV"
    "chguuN3hP/Kjm9PzwYhQ9jaPwon8n8QB85fHAXD0whZqipQ6qgYwBBZ7wWt3olQVtbCw99lwUARqIBbVTp2eb5Bh9MRwIKBz"
    "Ae1HFRDHXrsiCZEcYXzR0C/lHaiE/tbin4pEFq33EuftOxaooQBo4E65VRcvmNLhX//5f08cy9yRExE7RsqTMax9FQtY9UXj"
    "uFBVvaBpepBaC6Kn/kBVO2eYdPlZhcZqeXrp89QfprBT0Pr8DmM/WYB2An+EyXgdHgejD1So1KGWpaCZaWGtAVmLUvgLZyXe"
    "ClG30BRY7FGt18LyFw7k9V8eeYO//vP/cd230Ne62C6bLSSaQhn/sLdjd45OleOgdu/G7k7sYjECyS/rdMZdQTGE2E0SC82n"
    "GSfLPNWE5eyj2oFpS0Ki2vOikIg11cQsfTCIDr4+iIi+61UUWM7QM02GSiUQZ4Qvb9ujbtMUB3S31y16WCcHLJKO8m5yupuV"
    "raldzOxsJrqeiO5Au0dwqj2o3IXSDjdXzzNsdi+sTnAvkuFJRglXHEq1UMWLPVYqVsLAG53U702r9mb4Vi2ekx7LeCRcUfFN"
    "d6uynhDTFeKgXvvP7mQ7+uyOZG21MeYwjoGl164MKbspEDP2VPBzsIZbGMZe1WbnK/aNqtQHdSPdwo+iwCQGpy3US54wSjWl"
    "DJIpe++LMiS25QbD5yV6ivA6tawzpV6ylnkBO7yZLTJUMCOMhqgmEgLDk/MdloUot0vzCEGVaGoXvlPzgtgpUCNLyhEpVhxP"
    "DXk8jR24JHh8LjltPvAU85H8KRcKU7u84ruZV0JDespSwo9g30xTq2sEnouJ0Rc9tbq94DHfUpYFprtETDBsKYqp7v+Snweo"
    "Gb4wasZTynmfhg7UhLc8LPB6KzJ9puSEmFoNZDEtfIlBqam1Hww85mnJgdJ0ZBMIWz5qqneBkZ8HKBm9MErOpz7ddmWPcHV9"
    "8bjHZjE/LaUMnFIp1Klvv7Qr58bv8Xyc2sUwPudJ2gNS36AWYPyJa8jl+sEAbeMXJJnDqWhsFbl6PhqxVvfCwCnx7lTYNMTR"
    "7nQpJZvqim2KOL3Aq1My9mfdx8lQiw4ndPTI4zAirCtb92Q78ELywvZ5NF2Qvsoe2WIZxXLrNcwN64ZMmoNX4B7dZXj/a7oQ"
    "lzjCafBG4ZC/vwRDg6LHfjD103dCnWrKQAnNCBGypaxMQ1txn4ejh7M6I8amtkz1qi7y8wDLzE/JMg/Np78uPn9GUSmam1Bz"
    "rxTXTgBvS8FCcv1BKU8JGfigEEO5qfZdTKT/E/R916ver2sQdDlD9mvFsUX1gTXxxAhZhSlDKqRTzByg7GwCvJU6XlfYpKSg"
    "xaSghXY9EB6rqHWsP1fwuuAxAfNSpwNdEUAuWmL66Aw0rhMFpcB79Y0MyMi+//GqEBd2vgKe+ZR56/z+Z9hOYNJQ4w4aoBUp"
    "bIY0L39wcdeZrFYXQ8UiNFORFh1Es25pYdIdkKRR2nYY59w+DXJ9Tj1HTJR6wwwOPY4gDRF3jxDxzOL55NSR8fRp0GtafKOJ"
    "d9yt97/S4geYVIybNhGT4G6ktJM+iyNUjIHYbnHEpsXxAAz0IhQGbN6kTAyT0kFBMeKwPbGcR9BYj4zTTUXEeuqmsyp+GOob"
    "orhysI4FTfVm+lpcU3lDRr1WPk4vSCl6fOVYklI2MbugAbg7p9aoHjzu02GPQ48YZAxi7LEx1eN/8vMA0/gv7bRAsYs13B15"
    "RjQDVpRDGC+UQ7Ab31A+s6ZeDJAiuqABtH8D+9nRyyNu6DjwhI4If5AExkHiwhPByQ+SUzQnqeueJjO1EOc78g/ANb7lvsBu"
    "B9tsLziF7UZiefehGkgg5X6iKiBJI/t9FBN+UKfSsZWS2QYlxyBCPU97UOipl8w1LTHvUhn9D4h3wufxmFgrazc7SF+1xZSt"
    "dQOCfnP/I3T8KhshLzrPedQ7lZYQnftWr0jrwNBOf7lhIW9gIQ5xFHdWW6UTdnRERveJjEGDt1yMK1dTB83Tj57NWuF6LEBl"
    "H7Tr4IY0uflh3fbZZ4ryZZ/vfwXnQX2X/JoaG2XkSkK4cmuJ8e3F8awep/bHc2qEq+xTEgIBmQvRm3pHJtOIk0Io/el21RR6"
    "A2NzUmiMWL0jI51+zZ2HK5LZeSqof3SfVHJBriOTXNHieUzDTeE1FxkVvzhldw7KLnqrw+Mip5V6919RNS+oelckmyR1PXHh"
    "NBTuLixGDesErO9Zs039UdFSF3R0z5sdEVab1XXZoUyIyoyi96ayVUusBP6DlSXmJ3K5kdaCLiUoO77OqMxEFMz0ku1w9mAu"
    "YPyKWhAXy+KaqlAcsGxAtc1R3cEaKsVNqdWu0LSZeuCWTZohAMqim77T2USr0ohYwr7Hp5TqYK/TN4t5dcURJN0X6QDjEyti"
    "VjHxZ7EDg7hdCX4Qb4zt64rFKOj3iFoUUDLiQI0BVgfsgYIrlSpdgaC4SVl3X6cvF4nWLPMf0Cfb/gLqWQe9IzH1xysv94cb"
    "N1dbPeZCzv9eIoGXemYza+qNOqwNyixLqPVwIUZRzSQ5DVBbrV2hBv8yBf8LepGCP6JExOyV/a7rqz5teejX6MrT+N9BKZsj"
    "fgF/efbVyNuzlesN6ac47lDbQr8weP7Lj6g2ISev6PTDkijoC4ZfSH1/cjoYQYcodqicoUhYCorO407z1zpp8Bp7yJeC5sNc"
    "TBFBCftCTiuOGSJ8rLNxU1ZHHYbpfgJoPcF7MZDqQzLNrxKu+dVI3NR4w0is2IAn0xsrGe6NRawWnRGroVy8/wV077ItEOWn"
    "M6VrUgZ6ilEnii7MgFhCVsSv9XJQYqSoK+bEX2IqBHtBsImEIUqxJLTxFhzdEpFVKUbAFTIKBTAVLP25xKqPk3qx0LioFz4r"
    "dDUkWnk7VMySOCY+K3lb62NbedZkkOipO5Kg/itsrY+iF9cGMAHrGx1FnGzVnUpioDbOpUc1tdNtRQ2Vi6JStJgWLoRMpGpn"
    "faImiltRE8VWYQmPe5pXuyEzJPFMNNjaubEu8SIXJS45KVM9zOaTWxGXyp/RPkzjGe9ZwncUnXgl8xvEcmsJVTAN+bip5YhV"
    "/07jjso9ND31TNMnZm8B6vmpqQth5mdKGHFABrM+wgwZK1pFv6CW19QuJrRnuM265HEAQjFjc3rWf2JktLkI8fT5jmS+C5ow"
    "duSxrHZBFUcrluve0I8XpSbAxQILSEuHOubf/OevSKT7i1cP1hlhrguJgSDBgjqCzCy9iOTrqj4mBr6lARiXWTCl9uiLYSWR"
    "R0atq+eIBhsgWtgL69FzB4ZaPKPTheGoViyjk+q6FObysqxgfUAVIr9cRK5HOAxh0a4OmJfCyHg4g0DMwTr32s96S+NmjClI"
    "RV5iLARpGjlK9TRqxWkavbL3vHrV12PZGaEhyW6iwrfCemW0wuoNHCEnOd4ifxr50WUFCA/A78QPqA5RFHK4k6JQnBKxvOHM"
    "kd6h4jAI7S6PcPaKbYRoAh2iqQmOFzevRssbx7vjTyZvQAwDwTLoCoUkBOGRY+Ifpn/CUsIhIW5/wVKAcZBYXBEfD7R7jCMC"
    "swT3pZOAke9eyOnfS3DCdu9PtaaFh8QBcPbWngahlte+KD929Kn5in7u280EP37VJ5r6fwaTou3yA4aY1USdtVsN08J30GD9"
    "4Dw1WGnh92qwQ7rlLNZuwNYqxYwcBYwnq6PBOgAR113N6XmXlOGJciBSeHYa7B6vMFNYXqqwVNK49dkL91THG4pnbadmNPG7"
    "d//TaBenhCncP3SUJw4+zmbce/G+6HA54/pqjnRtMjixvk0Wm56Om1bllZ478FJ0dry0OeDxeF00Odbk2v2UXWLv7H0uSu0B"
    "j1yuZcsjcRKTIZS+0lKelEEQgV1V9z+Ok1BB65ieibqfnBkkh3iZKHlQ41ZP8darS3zPGdaiLxSEnUSPFT/DMhqHGV8Ov4ue"
    "UhPfjdfi0xtKD8+2VP0kgfBgRXGPnySY9YoAf6JdzJd+QBHXcGsZ7wqEQsfG9CdqFZ8vDyNyCeDtjM0uSU8U8ICLKngnXeqs"
    "daEJ6j1TX9xBjTXQ/R+YoSReZTb+d2tk5zpOKn4J7Y5bTMePLS3dxJnt4/3MbHNJ46jzxC29S6GonqYWOqmdoGuywgEuv4VH"
    "NbIgHnBRB53Jf7rkImWlwtu2ulufU0JZszoLUKGrmARV8ZArcoOvLDgPg212fNecl6NNM7lriRuiTnjyHfooJApu6xZiPR1V"
    "BXt3uDu4rYv7cmahpN6co9mcaR/DgrKZbg2QPgLq+8tle0Vnv71uvakoZRs43TRQ9wEKqsjYYyL7OmIbXNJhlvalaXYh1PtM"
    "m7uPTiP7VRKl4sen1LFXRBRqxbeyoVbIkatWDFOT+2NamT7/SyzIw9LNINsgJNs+a2D00W9UFXaiXvIiqBe9RvIFb2SDgdEC"
    "LcLFe0/JyrbuWyMPSmdIqr3xSOfk/EXQb45yjlphkvrGCbuvrqrDdVPf9FYYZk0zXprmvVBrfOygac9ojuh7SN2my1RgKmdo"
    "49HOYs9Vnb6j0dAzmfsQ6D189bPXgfjp09ghI33ozQ0JxRoBQxtWJlENEigZbgRkj7RHAiRHUtc4N/d2ItWXILK2T9CUGN9+"
    "rzNwNE1URNStr6DQo0H5PXc3jaubQW+8jFhjccpz4cFc8e2h2EmfFSiX93+B3w5ZUKKVJVYGuJNLUbeBQ3sWGu/54l3NcSVG"
    "i7lgKMMskNr3lUw5zXEH46B3noIGKd4Ku7REFSbm61vRTw2OLL+9v9Iq7KFv6S7sYROVVZe8gvp48/u2bl9saW0vsDRUBbff"
    "gYOO+AWa/eyKO8ukyg9oWTUcPsacYrvPL8NQ38ca2kHTIKfHWgu6vogpCcCY69VA0iKUR0HMSSo/Zmy2HoCdvg0GqD6Z5Mej"
    "AmVETv9MyQkSBlrVQP5lRvvjjm6UEF+/pw+RKY9ESBIE2e2HfpMB1QRSAZtf5cnrggXej4KrlmPMde81HodUYSbH0AdeNkD8"
    "wjeO+ukxaPWZ9j0InVSZ9YPz55ygYR2stY9/kEZ7BOvs9sX+IBsk5Vrba9NBRCc5Xf/3VZW3UVMvpFpNLyhFR8NGp+H0HnkI"
    "RByF4zeOKvPRPwencf4hNiZ1+AknVbqdQszP65lK6IfRSkZDgiq6fTq/Ig/r6RyLfvToirEr9f4T3pqle05ZU19NJgls1bt7"
    "TZ9oNSFAJhNQf2kxB68JKy/V427BOm3NGNpfi4EkI7nRwoAJ09IXtgyAkRj0bHIBm29wYY/303MHzojPfmOjt4HUcWZXw39p"
    "NW8x4bfEzj+4z6g7OWUA43lcimlQi701XNCuG3o0mNoPfcK9fJWXI+Kw8PYm3+OHLtUWp7t3BqoFXA4qPUol3l5B7LyCQv+U"
    "nuUhfHH5DJ5eldT+sH0GV0o/lZoSAYkc9z9Wecbf26avpZ8FIcLBACCpVqUAAT5/Ma/Z3iClzWbtV5HUSNW9DLqXt93htu0N"
    "JES2h4UdbBvySxy4wUV7fJmMAGfJ/c/0Ds8N7Tt20hAZXguQbwWYN1olffxatPElNzsV95jVkY/uTbGZRLrDHkchIurm96DA"
    "3hJT8Wj1uJkDv4SPzS9jrx+CI6+QC/WODDf5YUe8s8X6kFe0N7BR31z+rSoFxGKYvw91A/KqHv6BLyjSACBUC6rjIYjWoiKd"
    "cXglVEJUjhO6J/o6ok6uc1Lp6YlPGReJ1Y2RxHKebkGR49ChU2YIETbJVMWmzTWNHfgneo7zhssybdenTijNPOQzSg6eeZgz"
    "Jnclcg9+m633h4rO8UT7+gFSSMECxr4m+CgMCNIb0c6lZZ7XPiXoJW/qvilJ00GFuzan/w43EeWCFE7Qjw5/76EHJ+fbEMRP"
    "8bnwk3FNzjzPmy5Hy8OdVP3o4FloKzFCUqltk+gKEc5pNi5e1E24umyh8++wQLLDa/oTS0gsxbvgWjq76CnJk2itRyZ6qjtn"
    "ef9ztcR9Ax/hVUg7jnVmh8JpWHLy//1PqufEOhu+YaaF5YehsOI0OXTpa0mOs+E3oqtd/YmqAM3TiVWu4PNjzgceHRw2hAjb"
    "LKQfm9bMgfd6o3nEMvPnUlX6uAX/V26vwO23EksJtn0BS7n9iCUS0P/W8M38KL75Vgew7sxfa5MK2UyVtqYYtnuIwond6qEv"
    "eg6RoxBpVNouCqwya8Wj1Wrxa4cDJj1PQaJtOkaGNPaRm8KhnuQ0SVd4ODX0G2f0slDHtfPrN3kHq7kTqRfP5gHhLuwjeIvd"
    "G9e/FmOPc09XNSmwb66GUGRP7P6I643YU7ju1A4QDWZf/bp+fpg52CROofQnExRGSKxcU8OMeg9d0uYHO6QqkADEJpGjCmmZ"
    "C4CImX77xVQdozr7NbHu/yQyBPX9r5fA7l26D0BtFEgVHmt39gBtzVCHI8Ap3P5kJP+dasVjRdDdAa5Q0yU0sOeF6Skbu9Z+"
    "DbLReV9GyXu9dGA6oEGnxREuilHwWq/EI/og/OCciE9+iOyaqqoILwQa2zvpgrgz/AwdN0R0OjeERALtfWHwb7I7kDtvhUsE"
    "kxErYea/aV0BqmfiNfXNTpR3cDun9R9cV/fj3BNHIUsei2E0+ViZk9sicGDB8Fy0ze9zj1J0oV4k+D+zilbX4oinek8dNzxd"
    "How4T7yqoawkALXI95bvYSXepQIzVG4mpyBZ45O4cPPMBzMK5zd+8ZDki/XGQZJ6jBFzHCqtb55Dgj3YUptzPkkduCo6u4Ot"
    "8SKqKp845hD2XnAGZgXZD7JGqGG554Kabh/YqP13osfhPgcfuLja+V0BQOEHZpA20Q4XXgn4960X0d8LpBDmIGtFm79VQeJJ"
    "Aw22yYdhFX9xjyl8EsRwyh6U2Ku6vGnsIivj8/HXYzLED1LnNdxE9YH0DlipPoUotE67dd+JDUql1FVXhuj8bpNp52HHbJFi"
    "RjWcYtGZTBb8d3TPu+nRozDCuTq4OPvmbTr2kGsephxz4RsQWIE9AIm2f0QegaoA+O7Kwk7+7LTlwpXCZqIaGeV/Ns0GMNVI"
    "NGcQ+StgG02gM0M4gayVaILJ39Fk1i2RVxJ8JNXuYE7hkBW7hbwlAQovAE0wPIfA8F9RY3sS96QcqY0nAAM5GQ6i0kCUs+sH"
    "1jOMvuDTYl3wsiUiuWLEevhqtFqnXv1kgAWdcmNPyX0jKntflutd1tZajvEBLHJV/pDt8BzLPh/WVJ0kgPr01Ur2NgD1CRpz"
    "V5kMVsIQVEeg30dV115ei5PAU38LLju1Z80zmvcD26W6KSFDyXM4jtDCw+xtk8mvFVb91gBFmwYz8D5CL/gqu1xTmwssUS4e"
    "X5e7Ah9ejOtl7YqYFJMPQemEHa+JC8NH5MKTlcRSTy3qPyLulWF3XyrjfFffxOo5u+D/tDa4kpSS4uMZEUR1rRLHI6uZI47q"
    "Uge+VWn3mTvCA4g4nFF2FFg/pH6F2E8dmCI6W6ZQpBmsyqZcZVRcC06qHw6oj0OSeUz/mwNTLHgbjkaqmf14AtENth0Vfyz2"
    "ucITWAwNStBm1OcAPdk3aCVSfw7U72gcHnk7PPJu8s/vPCgv+DPNuW5h2VXzjk3nghoOOhVSrHI+Rh8ntomfnG2OdF7j78de"
    "VVuDe6J3mPLrA/v4eD8uCHsyrUBfRJcXLiIaj14JYnArc7nQTxrX+1QXRzVclcBLwnKtdtVqcr6wSpo9sBFb+jc54yQl0XHY"
    "8FUYx90tQ55JzkELsrHLV9DMOtt+Fl6nFQYFUWXOCxTXVbYb0mhKuj1OgyOGi/5jhh5UWEe4yidrfAhRT9TC/g1bciI2ikG/"
    "YuPbmxv7ZnPj8WAZtcXv6WjsGx2N/aGOxsQM82exyo7hCDxAt9A7xEmxzdZXGXFNDyMENopc0CgPq0VCdEW9C2+jk37Ad6nl"
    "1uPYFRm6D38cGifraExMkz71CTK2KSV5hmR9xxms+/JTtqMkcTySyaSqraddRjXuM/HSWOMJp+I1niONjnSq1fgb7lKMA6gX"
    "ulVqweOOwvMQzNw6Jys4sRXjprrwkp8HeG/x4qyn2lUCPpLZO3KO3IEOOJs4e35gnPSqYJ2CmbjHOXDNDN+oTZVoYr9+MOja"
    "YQAzt8tMkKz/ZtR1UaC5kZHxCELqZBYS0L4CD2ypK7loM0NZ0YAoz3uZxUhVMRRRTFH4pX2I0UytWwoEyGO25IHf7WMzCpDo"
    "mNKCOKEq6lbm4LkECcRCSYRsMfIOKQo/5LiJZSNWLAuANV1RYoCPyJQmI+RIW7SHRApYlz0mUOOGQSNUQ4FEG/AZGFRQNgkd"
    "ND40LBMmiHm892u18RTXkURcc8wLy9VVoTkhpowi3I/jSbUdP3jpNrZca1SAuHADLH5UW6qG10KzV023IGqr0nrV2Op11CZV"
    "sCFSQ/6cBgdK6jChn8zh4ofnajxj9ApSN66hFDdIb+i0gYEp8KoA7YHwkFJVol6XbykGdf/rWnR/WBbbZSk6HK9sbTiUpRYT"
    "YIqHfP3DsPuXxmBKhPDJ+6hNwP9nZtCgU6ixQQmXiDro0JVtm3orgPCe334crGUiGeisItui0CqzDrwUnbP0e6Az5vd/r/pj"
    "Ei77UOxgq28EfMqgh5JcAE5MhmSW1RMTm56YMQBZ0cP7YGLDBxO7+GD8+FzPlG5DT8guAYkFS4QcADmZbTdPWLKP97/uIQrX"
    "BCbXMDdNIIeM0YrkxJky74MNbaXfZjzVcnsG7NlYjsBUCbuRHR5vZJtYOdjXBj58gDI24pOMhPv//4ObqfNRAQA="
)

if not RUTA_DATOS.exists():
    RUTA_DATOS.parent.mkdir(parents=True, exist_ok=True)
    RUTA_DATOS.write_bytes(gzip.decompress(base64.b64decode(_BLOB)))
    print(f"Corpus escrito en {RUTA_DATOS} (copia embebida).")
else:
    print(f"Se usará el corpus existente en {RUTA_DATOS}.")

sha_real = hashlib.sha256(RUTA_DATOS.read_bytes()).hexdigest()
print("SHA-256:", sha_real[:16], "…")
print("Coincide con la versión embebida:", sha_real == SHA_ESPERADO)

### 4.2 · Carga y particiones

Las particiones vienen **fijadas en el archivo** (campo `split`), no se
calculan aquí. Es una decisión deliberada: si cada notebook hiciera su propio
`train_test_split`, cuatro arquitecturas estarían evaluándose sobre conjuntos
distintos y las métricas no serían comparables entre sí.

La partición es estratificada por categoría (12 entrenamiento + 3 validación
por clase), generada con semilla 42.

In [ ]:
import json
from collections import Counter

registros = [json.loads(l) for l in RUTA_DATOS.read_text(encoding="utf-8").splitlines()]

train = [r for r in registros if r["split"] == "train"]
val   = [r for r in registros if r["split"] == "validation"]
demo  = [r for r in registros if r["es_demo"]]

CATEGORIAS = [
    "suma", "resta", "multiplicacion", "division", "operaciones_combinadas",
    "potencias_raices", "fracciones", "porcentajes", "ecuaciones",
    "geometria", "estadistica_probabilidad",
]
CAT2ID = {c: i for i, c in enumerate(CATEGORIAS)}
ID2CAT = {i: c for c, i in CAT2ID.items()}

print(f"Total: {len(registros)}  |  train: {len(train)}  |  validación: {len(val)}")
print(f"Ejemplos de demostración (todos en validación): {[d['id'] for d in demo]}")
print()
print("Distribución por categoría (train / val):")
ctr, cva = Counter(r["categoria"] for r in train), Counter(r["categoria"] for r in val)
for c in CATEGORIAS:
    print(f"  {c:26s} {ctr[c]:3d} / {cva[c]:2d}")
print()
print("Ejemplo completo:")
print(json.dumps(train[0], ensure_ascii=False, indent=2))

Un ejemplo del corpus se ve así:

```
entrada : "María tiene 48 caramelos y quiere repartirlos por igual entre 6
           amigos. ¿Cuántos caramelos recibirá cada amigo?"
salida  : "Paso 1: Repartir en partes iguales es dividir.
           Paso 2: 48 ÷ 6 = 8.
           Respuesta final: 8 caramelos"
valor   : "8"
```

El campo `valor` es la clave de toda la evaluación automática: es la respuesta
en forma canónica (sin unidades ni texto). Comparar `valor` contra lo que el
modelo escribe después de `Respuesta final:` nos da una métrica objetiva de
**si el modelo resolvió bien el problema**, independiente de cómo lo redactó.

## 5 · Preprocesamiento

### La plantilla de prompt

Todo ejemplo se convierte al formato de conversación que Qwen2.5-Instruct
espera (`<|im_start|>system … <|im_start|>user … <|im_start|>assistant`).
Usar `apply_chat_template` en vez de concatenar texto a mano importa: el
modelo fue alineado con esos tokens especiales, y saltárselos degrada el
baseline de forma artificial.

La misma plantilla se usa en el baseline, en el entrenamiento y en la
evaluación posterior. Si el prompt cambiara entre fases, estaríamos midiendo
el efecto del prompt y no el del fine-tuning.

### El enmascaramiento de la pérdida

Este es el punto técnico más importante de la sección. Un ejemplo tokenizado
contiene el prompt y la respuesta. Si calculamos la pérdida sobre **todos** los
tokens, el modelo dedica capacidad a aprender a predecir el enunciado del
problema, que es exactamente lo que no queremos: en producción el enunciado
lo escribe el usuario.

La solución estándar es poner `-100` en las etiquetas de los tokens del prompt.
PyTorch ignora esas posiciones al calcular la entropía cruzada, y el gradiente
se concentra en la respuesta.

```
tokens :  [<|im_start|>system ... usuario: 48 ÷ 6 ...][ Paso 1: ... Respuesta final: 8 ]
labels :  [ -100  -100  -100  -100  -100  -100  -100 ][ 30821  25 ... 23   ]
           \________ ignorado por la pérdida ________/ \____ aquí se aprende ____/
```

In [ ]:
INSTRUCCION = (
    "Eres un tutor de matemáticas. Resuelve el siguiente problema explicando "
    "el procedimiento paso a paso y termina con la respuesta final."
)


def construir_prompt(entrada):
    """Prompt de inferencia: termina justo donde el modelo debe empezar a escribir."""
    mensajes = [
        {"role": "system", "content": INSTRUCCION},
        {"role": "user", "content": entrada},
    ]
    return tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )


print(construir_prompt("Resuelve la siguiente operación: 36 × 24."))

## 6 · Tokenización

Antes de fijar `LONGITUD_MAX` medimos cuántos tokens ocupan realmente nuestros
ejemplos. Elegir el número a ojo tiene dos costos: si se queda corto trunca
respuestas (el modelo aprendería a no terminar nunca), y si se pasa
desperdicia cómputo en padding.

También calculamos la **fertilidad** del tokenizador: tokens por palabra. Es
la métrica que usaremos en el notebook de comparación para contrastar los
tokenizadores de Qwen (BPE, 151k), BETO (WordPiece, 31k) y FLAN-T5
(SentencePiece, 32k) sobre el mismo texto en español.

In [ ]:
import numpy as np

longitudes, fertilidades = [], []
for r in registros:
    completo = construir_prompt(r["entrada"]) + r["salida"] + tokenizer.eos_token
    ids = tokenizer(completo, add_special_tokens=False)["input_ids"]
    longitudes.append(len(ids))
    texto_plano = r["entrada"] + " " + r["salida"]
    n_tok = len(tokenizer(texto_plano, add_special_tokens=False)["input_ids"])
    fertilidades.append(n_tok / len(texto_plano.split()))

longitudes = np.array(longitudes)
print(f"Tokens por ejemplo (prompt + respuesta):")
print(f"  min={longitudes.min()}  media={longitudes.mean():.1f}  "
      f"p95={np.percentile(longitudes, 95):.0f}  max={longitudes.max()}")
print(f"  ejemplos que excederían LONGITUD_MAX={LONGITUD_MAX}: "
      f"{(longitudes > LONGITUD_MAX).sum()}")
print()
print(f"Fertilidad (tokens/palabra en español): {np.mean(fertilidades):.3f}")
print(f"Tamaño del vocabulario: {tokenizer.vocab_size}")

STATS_TOKENIZADOR = {
    "modelo": MODELO_ID,
    "tokenizador": tokenizer.__class__.__name__,
    "vocabulario": int(tokenizer.vocab_size),
    "fertilidad_media": float(np.mean(fertilidades)),
    "tokens_media": float(longitudes.mean()),
    "tokens_max": int(longitudes.max()),
}

### Inspección cualitativa

Ver cómo se parte un texto concreto explica más que cualquier estadística.
Presten atención a los símbolos matemáticos (`÷`, `×`, `²`, `√`): son el punto
donde los tres tokenizadores del proyecto se comportan de forma más distinta.

In [ ]:
muestras = [
    "Resuelve la siguiente operación: 864 ÷ 12.",
    "¿Cuál es el área de un círculo de radio 5? √144 + 5² = 37",
    "Paso 1: Repartir en partes iguales es dividir.",
]
for m in muestras:
    ids = tokenizer(m, add_special_tokens=False)["input_ids"]
    piezas = [tokenizer.decode([i]) for i in ids]
    print(f"\nTexto ({len(ids)} tokens): {m}")
    print("   " + " | ".join(piezas))

desconocidos = sum(
    1 for m in muestras
    for i in tokenizer(m, add_special_tokens=False)["input_ids"]
    if i == tokenizer.unk_token_id
)
print(f"\nTokens desconocidos (<unk>): {desconocidos}")

### Tokenización del dataset con enmascaramiento

In [ ]:
from datasets import Dataset


def tokenizar(reg):
    prompt = construir_prompt(reg["entrada"])
    completo = prompt + reg["salida"] + tokenizer.eos_token

    ids_prompt = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    tok = tokenizer(
        completo, add_special_tokens=False,
        truncation=True, max_length=LONGITUD_MAX,
    )

    # -100 en el prompt: la pérdida solo se calcula sobre la respuesta.
    etiquetas = list(tok["input_ids"])
    for i in range(min(len(ids_prompt), len(etiquetas))):
        etiquetas[i] = -100
    tok["labels"] = etiquetas
    return tok


ds_train = Dataset.from_list([tokenizar(r) for r in train])
ds_val   = Dataset.from_list([tokenizar(r) for r in val])

print(ds_train)
ejemplo = ds_train[0]
n_ignorados = sum(1 for e in ejemplo["labels"] if e == -100)
print(f"\nEjemplo 0: {len(ejemplo['input_ids'])} tokens, "
      f"{n_ignorados} enmascarados (prompt), "
      f"{len(ejemplo['labels']) - n_ignorados} con gradiente (respuesta).")
print("\nTexto sobre el que realmente se aprende:")
print(tokenizer.decode([i for i, e in zip(ejemplo["input_ids"], ejemplo["labels"]) if e != -100]))

Usamos **padding dinámico**: cada lote se rellena hasta el ejemplo más largo
de ese lote, no hasta 384. Con longitudes tan dispares (de ~60 a ~200 tokens)
esto reduce el cómputo desperdiciado a la mitad frente a `padding="max_length"`.
El *collator* rellena `input_ids` con el token de padding y `labels` con
`-100`, que es justo lo que necesitamos.

In [ ]:
from transformers import DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,   # el padding de las etiquetas también se ignora
    return_tensors="pt",
)
print("Collator listo (padding dinámico por lote).")

### Métricas de generación

Antes de tocar el modelo definimos **cómo vamos a medirlo**. Fijar las métricas
antes de ver resultados evita el sesgo de elegir después la métrica que mejor
nos deja.

| Métrica | Qué mide | Por qué está |
|---|---|---|
| **Exactitud de la respuesta** | ¿El número final es correcto? | Es la única métrica que responde "¿sirve como tutor?". Un procedimiento bonito con resultado equivocado es un fracaso. |
| **Formato válido** | ¿Usó `Paso N:` y `Respuesta final:`? | Separa *aprender a resolver* de *aprender a formatear*. El fine-tuning con pocos datos suele mejorar mucho lo segundo y poco lo primero; sin esta métrica confundiríamos ambos efectos. |
| **ROUGE-L** | Solapamiento de la explicación con la de referencia | Aproxima si el procedimiento se parece al esperado. Es una métrica débil (premia coincidencia léxica, no razonamiento) y así hay que leerla. |
| **Longitud media** | Palabras generadas | Detecta el modo de fallo típico del baseline: divagar o repetirse hasta agotar `max_new_tokens`. |

Nota metodológica importante: para extraer la respuesta del modelo usamos el
marcador `Respuesta final:` y, si no aparece, **el último número del texto**.
Sin ese respaldo estaríamos castigando al baseline por desconocer un formato
que todavía no le hemos enseñado, y la mejora del fine-tuning se vería
artificialmente enorme.

In [ ]:
import re
import unicodedata
from fractions import Fraction

MARCADOR_RESPUESTA = "Respuesta final:"

_PAT_RESPUESTA = re.compile(r"Respuesta\s+final\s*:\s*(.+)", re.IGNORECASE)
_PAT_PASO1     = re.compile(r"Paso\s*1\s*:", re.IGNORECASE)
_PAT_CATEGORIA = re.compile(r"Categor[ií]a\s*:\s*([a-zA-Z_]+)", re.IGNORECASE)
# La alternativa de fracción va primero: en "3/5" queremos capturar la fracción
# completa, no el "3" suelto.
_PAT_VALOR = re.compile(r"-?\d+(?:\.\d+)?\s*/\s*-?\d+(?:\.\d+)?|-?\d+(?:\.\d+)?")


def _sin_miles(texto):
    """Quita la coma como separador de miles. El corpus usa el punto como
    separador decimal y nunca la coma, así que la conversión no es ambigua."""
    return texto.replace(",", "")


def a_float(valor):
    if valor is None:
        return None
    try:
        return float(Fraction(valor)) if "/" in valor else float(valor)
    except (ValueError, ZeroDivisionError):
        return None


def valor_predicho(texto):
    """Extrae la respuesta del modelo en forma canónica.

    Prioridad 1: el número que sigue al marcador 'Respuesta final:'.
    Prioridad 2: el último número del texto.

    El segundo caso importa para que la comparación sea JUSTA: un modelo sin
    fine-tuning no conoce nuestro formato, y penalizarlo por eso mediría
    obediencia al formato, no capacidad matemática. Con el fallback medimos lo
    segundo; el apego al formato se mide aparte con `formato_valido`.
    """
    m = _PAT_RESPUESTA.search(texto)
    if m:
        linea = m.group(1).splitlines()[0]
        v = _PAT_VALOR.search(_sin_miles(linea))
        if v:
            return v.group(0).replace(" ", "")
    todos = _PAT_VALOR.findall(_sin_miles(texto))
    return todos[-1].replace(" ", "") if todos else None


def respuesta_correcta(generado, valor_oro, tol=1e-6):
    a = a_float(valor_predicho(generado))
    b = a_float(valor_oro)
    if a is None or b is None:
        return False
    return abs(a - b) <= tol * max(1.0, abs(b))


def formato_valido(texto):
    """¿El modelo produjo la estructura que le enseñamos?"""
    return bool(_PAT_PASO1.search(texto)) and bool(_PAT_RESPUESTA.search(texto))


def categoria_predicha(texto):
    m = _PAT_CATEGORIA.search(texto)
    return m.group(1).lower() if m else None


def _tokens(texto):
    t = unicodedata.normalize("NFKD", texto.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.findall(r"[a-z0-9]+|[^\sa-z0-9]", t)


def _lcs(a, b):
    """Longitud de la subsecuencia común más larga (programación dinámica)."""
    previa = [0] * (len(b) + 1)
    for x in a:
        actual = [0]
        for j, y in enumerate(b):
            actual.append(previa[j] + 1 if x == y else max(previa[j + 1], actual[j]))
        previa = actual
    return previa[-1]


def rouge_l(generado, referencia):
    """ROUGE-L (F1 sobre la subsecuencia común más larga).

    Se implementa a mano en lugar de usar `evaluate` para que el notebook no
    dependa de descargas en tiempo de ejecución y el número sea exactamente
    reproducible.
    """
    p, r = _tokens(generado), _tokens(referencia)
    if not p or not r:
        return 0.0
    l = _lcs(p, r)
    if l == 0:
        return 0.0
    prec, rec = l / len(p), l / len(r)
    return 2 * prec * rec / (prec + rec)


def evaluar_generacion(generados, registros):
    """Métricas de generación sobre un conjunto de ejemplos.

    exactitud       : ¿la respuesta final es numéricamente correcta?  <- la que importa
    formato_valido  : ¿respetó la estructura Paso N / Respuesta final?
    rouge_l         : ¿se parece el procedimiento al de referencia?
    long_media      : longitud media en palabras (detecta divagación)
    """
    assert len(generados) == len(registros)
    n = len(generados)
    correctas = [respuesta_correcta(g, r["valor"]) for g, r in zip(generados, registros)]
    formatos  = [formato_valido(g) for g in generados]
    rouges    = [rouge_l(g, r["salida"]) for g, r in zip(generados, registros)]
    return {
        "n": n,
        "exactitud": sum(correctas) / n,
        "formato_valido": sum(formatos) / n,
        "rouge_l": sum(rouges) / n,
        "long_media": sum(len(g.split()) for g in generados) / n,
        "_correctas": correctas,
    }


def tabla_metricas(antes, despues, titulo="Baseline vs Fine-tuned"):
    """Imprime la comparación en el formato que usaremos en el informe."""
    filas = [
        ("Exactitud de la respuesta", "exactitud", "{:.1%}"),
        ("Formato válido",            "formato_valido", "{:.1%}"),
        ("ROUGE-L del procedimiento", "rouge_l", "{:.3f}"),
        ("Longitud media (palabras)", "long_media", "{:.1f}"),
    ]
    ancho = 30
    print(titulo)
    print("=" * 68)
    print(f"{'Métrica':{ancho}s} {'Baseline':>12s} {'Fine-tuned':>12s} {'Δ':>10s}")
    print("-" * 68)
    for etiqueta, clave, fmt in filas:
        a, d = antes[clave], despues[clave]
        print(f"{etiqueta:{ancho}s} {fmt.format(a):>12s} {fmt.format(d):>12s} "
              f"{d - a:>+10.3f}")
    print("=" * 68)

In [ ]:
import json
from pathlib import Path

DIR_RESULTADOS = Path("resultados")
DIR_RESULTADOS.mkdir(exist_ok=True)


def guardar_resultados(nombre, payload):
    """Persiste las métricas para que el notebook de comparación las agregue.

    Sin este paso, comparar las cuatro arquitecturas obligaría a re-ejecutar
    todo en una sola sesión. Con él, cada notebook se ejecuta cuando se pueda y
    la comparación se hace al final leyendo los JSON.
    """
    limpio = {}
    for k, v in payload.items():
        if isinstance(v, dict):
            limpio[k] = {kk: vv for kk, vv in v.items() if not kk.startswith("_")}
        elif not k.startswith("_"):
            limpio[k] = v
    ruta = DIR_RESULTADOS / f"{nombre}.json"
    ruta.write_text(json.dumps(limpio, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Métricas guardadas en {ruta}")
    return ruta

## 7 · Baseline

**La pieza más importante del experimento.** Sin una medición previa no se
puede afirmar que el fine-tuning sirvió: cualquier resultado posterior sería
un número sin referencia.

Condiciones de la medición, idénticas a las que usaremos después del
entrenamiento:

- Mismos 33 ejemplos de validación (nunca vistos en entrenamiento).
- Mismo prompt, misma plantilla de chat.
- **Decodificación greedy** (`do_sample=False`): sin aleatoriedad. Si
  muestreáramos, dos ejecuciones darían métricas distintas y no sabríamos si
  la diferencia viene del modelo o del azar.
- Mismo techo de tokens generados.

In [ ]:
from tqdm.auto import tqdm


@torch.no_grad()
def generar(modelo, entradas, max_new_tokens=MAX_TOKENS_GEN):
    """Genera una respuesta por entrada. Decodifica solo los tokens NUEVOS.

    Detalle que suele producir errores: recortar por longitud de caracteres del
    prompt es frágil (el detokenizador no siempre reconstruye el prompt
    carácter a carácter). Recortar por número de tokens de entrada es exacto.
    """
    modelo.eval()
    salidas = []
    for entrada in tqdm(entradas, desc="generando"):
        prompt = construir_prompt(entrada)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(modelo.device)
        out = modelo.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        nuevos = out[0][inputs["input_ids"].shape[1]:]
        salidas.append(tokenizer.decode(nuevos, skip_special_tokens=True).strip())
    return salidas


gen_baseline = generar(modelo, [r["entrada"] for r in val])
metricas_baseline = evaluar_generacion(gen_baseline, val)

print()
for k, v in metricas_baseline.items():
    if not k.startswith("_"):
        print(f"  {k:16s}: {v}")

### Inspección cualitativa del baseline

Los números resumen; los ejemplos explican. Estas cinco preguntas son las
mismas que revisaremos después del fine-tuning.

In [ ]:
gen_demo_baseline = generar(modelo, [d["entrada"] for d in demo])

for d, g in zip(demo, gen_demo_baseline):
    print("=" * 78)
    print(f"[{d['id']} · {d['categoria']}] {d['entrada']}")
    print("-" * 78)
    print("BASELINE:")
    print(g[:700])
    print("-" * 78)
    print(f"Esperado: {d['salida']}")
    print(f"¿Respuesta correcta? {respuesta_correcta(g, d['valor'])}   "
          f"¿Formato válido? {formato_valido(g)}")
print("=" * 78)

> **Anoten qué falla exactamente.** Casi siempre son tres cosas distintas:
> (a) no usa el formato `Paso N` / `Respuesta final`, (b) se extiende de más o
> inventa preguntas adicionales, (c) se equivoca en la aritmética. El
> fine-tuning con 132 ejemplos arregla (a) y (b) con facilidad; (c) es el
> problema difícil y es donde hay que mirar con lupa.

## 8 · Configuración del fine-tuning

### LoRA: los tres números que importan

| Hiperparámetro | Valor | Razón |
|---|---|---|
| `r` (rango) | 16 | Capacidad del adaptador. Con 132 ejemplos, un rango alto (64+) memoriza; uno muy bajo (4) no alcanza a aprender el formato. 16 es el punto medio habitual para datasets pequeños. |
| `lora_alpha` | 32 | Factor de escala: el aporte del adaptador se multiplica por `alpha/r` = 2. La convención `alpha = 2r` funciona bien y evita tener que ajustar dos cosas a la vez. |
| `target_modules` | proyecciones de atención | Es donde el modelo decide *a qué atiende*. Añadir las capas MLP (`gate_proj`, `up_proj`, `down_proj`) da algo más de capacidad a costa de más parámetros y más riesgo de sobreajuste con este tamaño de corpus. |
| `lora_dropout` | 0.05 | Regularización ligera. Con datasets pequeños ayuda; subirlo mucho hace la pérdida ruidosa e ilegible. |

### Hiperparámetros de entrenamiento

Con 132 ejemplos, las decisiones se toman al revés que con datasets grandes:
el riesgo no es no aprender, es **memorizar**. Por eso evaluamos cada época
sobre validación y nos quedamos con el mejor punto, no con el último.

In [ ]:
import inspect
from transformers import TrainingArguments


def construir_args(clase=TrainingArguments, **kwargs):
    """Crea TrainingArguments filtrando los parámetros que la versión instalada
    de `transformers` no reconoce.

    Motivo: entre versiones recientes `evaluation_strategy` pasó a llamarse
    `eval_strategy`. Sin esta capa, el notebook funciona hoy y falla el
    semestre que viene. Se prefiere `eval_strategy` y se traduce si hace falta.
    """
    admitidos = set(inspect.signature(clase.__init__).parameters)
    if "eval_strategy" in kwargs and "eval_strategy" not in admitidos:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    descartados = [k for k in kwargs if k not in admitidos]
    for k in descartados:
        kwargs.pop(k)
    if descartados:
        print(f"Parámetros no soportados por esta versión, se omiten: {descartados}")
    return clase(**kwargs)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USAR_QLORA:
    modelo = prepare_model_for_kbit_training(modelo)

config_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

modelo = get_peft_model(modelo, config_lora)

# Los pesos base están en fp16, pero los parámetros ENTRENABLES deben estar en
# fp32: con entrenamiento en precisión mixta, mantener los pesos maestros en
# fp16 provoca subdesbordamiento del gradiente y la pérdida se va a NaN.
for nombre, param in modelo.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

modelo.print_trainable_parameters()

### Weights & Biases

W&B registra automáticamente la curva de pérdida, los hiperparámetros y el
consumo de GPU. Es lo que después nos permitirá comparar las cuatro
arquitecturas sobre los mismos ejes en lugar de sobre capturas de pantalla.

Convención de nombres del proyecto (idéntica en los cinco notebooks):

- **Proyecto:** `tutor-matematicas-arquitecturas`
- **Run:** `qwen-lora-finetune`
- **Tags:** identifican arquitectura y fase, para poder filtrar en el panel

Si no quieren usar W&B, pongan `USAR_WANDB = False`: el notebook seguirá
funcionando y las métricas se guardarán igual en `resultados/`.

In [ ]:
import os

USAR_WANDB   = True          # ponlo en False para trabajar sin conexión a W&B
PROYECTO     = "tutor-matematicas-arquitecturas"
NOMBRE_RUN   = "qwen-lora-finetune"
TAGS         = ["qwen", "decoder-only", "lora", "generacion"]

if USAR_WANDB:
    import wandb
    wandb.login()            # pedirá la API key la primera vez
    os.environ["WANDB_PROJECT"] = PROYECTO
    os.environ["WANDB_LOG_MODEL"] = "false"
    REPORTAR_A = "wandb"
else:
    os.environ["WANDB_MODE"] = "disabled"
    REPORTAR_A = "none"

print(f"Registro de experimentos: {REPORTAR_A}  |  run: {NOMBRE_RUN}")

In [ ]:
from transformers import Trainer

args = construir_args(
    output_dir=f"{DIR_CHECKPOINTS}/qwen-lora",
    run_name=NOMBRE_RUN,

    num_train_epochs=8,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,       # lote efectivo = 8
    per_device_eval_batch_size=2,

    learning_rate=2e-4,                  # alto a propósito: LoRA lo tolera
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,

    fp16=(DEVICE == "cuda"),
    logging_steps=5,
    eval_strategy="epoch",               # curva de validación por época
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,         # nos quedamos con la mejor época
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to=REPORTAR_A,
    seed=SEMILLA,
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
)

print(f"Pasos de optimización por época: "
      f"{len(ds_train) // (args.per_device_train_batch_size * args.gradient_accumulation_steps)}")
print(f"Pasos totales estimados: {int(args.num_train_epochs) * (len(ds_train) // (args.per_device_train_batch_size * args.gradient_accumulation_steps))}")

## 9 · Entrenamiento

### Cómo leer lo que va apareciendo

- **`loss`** (entrenamiento) debe bajar de forma sostenida. Si oscila sin
  tendencia, el learning rate es demasiado alto.
- **`eval_loss`** (validación) es la que de verdad importa. Mientras baje, el
  modelo generaliza. Cuando empieza a subir mientras `loss` sigue bajando,
  eso es **sobreajuste**: está memorizando los 132 ejemplos.
- Con un corpus pequeño, ver `eval_loss` tocar fondo hacia la época 4-6 y
  repuntar después es lo esperable, **no un error**. Para eso está
  `load_best_model_at_end`.

In [ ]:
resultado_entrenamiento = trainer.train()

print()
print(f"Tiempo de entrenamiento: {resultado_entrenamiento.metrics['train_runtime']:.1f} s")
print(f"Pérdida final de entrenamiento: {resultado_entrenamiento.metrics['train_loss']:.4f}")

In [ ]:
import pandas as pd

historial = pd.DataFrame(trainer.state.log_history)
curvas = historial[["epoch", "loss", "eval_loss"]].groupby("epoch").first().dropna(how="all")
print(curvas.to_string())

mejor = historial.dropna(subset=["eval_loss"]).sort_values("eval_loss").iloc[0]
print(f"\nMejor época: {mejor['epoch']:.0f}  (eval_loss = {mejor['eval_loss']:.4f})")

In [ ]:
import matplotlib.pyplot as plt

tr = historial.dropna(subset=["loss"])
ev = historial.dropna(subset=["eval_loss"])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(tr["epoch"], tr["loss"], label="Entrenamiento", alpha=0.8)
ax.plot(ev["epoch"], ev["eval_loss"], label="Validación", marker="o")
ax.axvline(mejor["epoch"], ls="--", c="gray", lw=1,
           label=f"Mejor época ({mejor['epoch']:.0f})")
ax.set_xlabel("Época"); ax.set_ylabel("Pérdida")
ax.set_title("Qwen2.5-1.5B + LoRA · curvas de pérdida")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Tras `load_best_model_at_end`, el modelo bueno es `trainer.model` (el Trainer
# recarga el mejor checkpoint). Guardar y evaluar desde ahí evita quedarnos con
# los pesos de la última época, que no son los mejores.
modelo = trainer.model

modelo.save_pretrained(DIR_ADAPTADOR)
tokenizer.save_pretrained(DIR_ADAPTADOR)
print(f"Adaptador LoRA guardado en {DIR_ADAPTADOR}/")
print("Son unos pocos MB: LoRA no guarda el modelo base, solo las matrices A y B.")
print("El notebook 3 (pipeline Qwen+BERT) cargará este adaptador.")

## 10 · Integración con Weights & Biases

La sesión de W&B se configuró antes de entrenar (sección 8) porque el
`Trainer` necesita `report_to="wandb"` desde el principio; lo que hacemos aquí
es **completar el registro** con lo que el `Trainer` no sabe: las métricas de
generación, la comparación baseline vs fine-tuned y ejemplos concretos.

Lo que queda registrado en el proyecto `tutor-matematicas-arquitecturas`:

| Origen | Contenido |
|---|---|
| Automático (`Trainer`) | `train/loss`, `eval/loss` por época, learning rate, gradientes, uso de GPU, todos los hiperparámetros |
| Manual (esta sección) | exactitud, formato válido, ROUGE-L antes y después; tabla de generaciones ejemplo por ejemplo |

La tabla de generaciones es la parte más útil en la práctica: permite abrir un
run tres semanas después y ver **qué** contestó el modelo, no solo qué número
sacó.

## 11 · Evaluación

Volvemos a ejecutar exactamente el mismo procedimiento del baseline, sobre los
mismos 33 ejemplos de validación, con el mismo decodificado greedy. Lo único
que cambió es el adaptador LoRA.

In [ ]:
gen_finetuned = generar(modelo, [r["entrada"] for r in val])
metricas_finetuned = evaluar_generacion(gen_finetuned, val)

print()
for k, v in metricas_finetuned.items():
    if not k.startswith("_"):
        print(f"  {k:16s}: {v}")

### Desglose por categoría

El promedio global esconde información. Un tutor que acierta el 90% en sumas y
el 10% en ecuaciones no es un tutor con 50% de exactitud: es un tutor que no
sirve para álgebra. Este desglose es también lo que motivará el pipeline con
clasificador del notebook 3.

In [ ]:
from collections import defaultdict

por_cat = defaultdict(lambda: {"n": 0, "base": 0, "ft": 0})
for r, ok_b, ok_f in zip(val, metricas_baseline["_correctas"], metricas_finetuned["_correctas"]):
    d = por_cat[r["categoria"]]
    d["n"] += 1
    d["base"] += int(ok_b)
    d["ft"] += int(ok_f)

print(f"{'categoría':26s} {'n':>3s} {'baseline':>10s} {'fine-tuned':>12s}")
print("-" * 56)
for cat in CATEGORIAS:
    d = por_cat.get(cat)
    if d:
        print(f"{cat:26s} {d['n']:3d} {d['base']/d['n']:>9.0%} {d['ft']/d['n']:>11.0%}")

EXACTITUD_POR_CATEGORIA = {
    cat: {"n": d["n"], "baseline": d["base"] / d["n"], "finetuned": d["ft"] / d["n"]}
    for cat, d in por_cat.items()
}

## 12 · Comparación Baseline vs Fine-tuned

In [ ]:
tabla_metricas(metricas_baseline, metricas_finetuned,
               titulo=f"Qwen2.5-1.5B · {len(val)} ejemplos de validación")

In [ ]:
gen_demo_ft = generar(modelo, [d["entrada"] for d in demo])

for d, antes, despues in zip(demo, gen_demo_baseline, gen_demo_ft):
    print("=" * 78)
    print(f"[{d['id']} · {d['categoria']}] {d['entrada']}")
    print("-" * 78)
    print("ANTES (baseline):")
    print(antes[:500])
    print("-" * 78)
    print("DESPUÉS (fine-tuned):")
    print(despues[:500])
    print("-" * 78)
    print(f"Referencia: {d['salida']}")
    print(f"Correcta -> antes: {respuesta_correcta(antes, d['valor'])} | "
          f"después: {respuesta_correcta(despues, d['valor'])}")
print("=" * 78)

### Análisis de cambios individuales

Cuántos ejemplos pasaron de mal a bien y —esto se suele olvidar— cuántos
pasaron de bien a mal. Una mejora neta pequeña puede esconder mucho movimiento
en ambas direcciones, señal de que el modelo está inestable y no de que
aprendió algo sólido.

In [ ]:
mejoraron = [r["id"] for r, b, f in zip(val, metricas_baseline["_correctas"], metricas_finetuned["_correctas"]) if not b and f]
empeoraron = [r["id"] for r, b, f in zip(val, metricas_baseline["_correctas"], metricas_finetuned["_correctas"]) if b and not f]

print(f"Pasaron de incorrecto a correcto ({len(mejoraron)}): {mejoraron}")
print(f"Pasaron de correcto a incorrecto ({len(empeoraron)}): {empeoraron}")
print(f"Mejora neta: {len(mejoraron) - len(empeoraron):+d} de {len(val)} ejemplos")

In [ ]:
RESULTADOS_QWEN = {
    "notebook": "S04_Lab_Fine_tuning_Qwen",
    "arquitectura": "decoder-only",
    "modelo": MODELO_ID,
    "metodo": "QLoRA" if USAR_QLORA else "LoRA",
    "n_train": len(train),
    "n_val": len(val),
    "epocas": float(args.num_train_epochs),
    "learning_rate": args.learning_rate,
    "tiempo_entrenamiento_s": resultado_entrenamiento.metrics["train_runtime"],
    "train_loss_final": resultado_entrenamiento.metrics["train_loss"],
    "eval_loss_mejor": float(mejor["eval_loss"]),
    "mejor_epoca": float(mejor["epoch"]),
    "parametros_entrenables": sum(p.numel() for p in modelo.parameters() if p.requires_grad),
    "parametros_totales": n_par,
    "baseline": metricas_baseline,
    "finetuned": metricas_finetuned,
    "por_categoria": EXACTITUD_POR_CATEGORIA,
    "tokenizador": STATS_TOKENIZADOR,
    "mejoraron": mejoraron,
    "empeoraron": empeoraron,
}

guardar_resultados("qwen", RESULTADOS_QWEN)

In [ ]:
if USAR_WANDB:
    import wandb

    if wandb.run is None:
        wandb.init(project=PROYECTO, name=NOMBRE_RUN, tags=TAGS, reinit=True)

    tabla = wandb.Table(columns=["id", "categoria", "entrada", "referencia",
                                 "baseline", "finetuned", "ok_baseline", "ok_finetuned"])
    for r, b, f, okb, okf in zip(val, gen_baseline, gen_finetuned,
                                 metricas_baseline["_correctas"],
                                 metricas_finetuned["_correctas"]):
        tabla.add_data(r["id"], r["categoria"], r["entrada"], r["salida"], b, f, okb, okf)

    wandb.log({"evaluacion/generaciones": tabla})
    wandb.summary.update({
        "baseline/exactitud": metricas_baseline["exactitud"],
        "baseline/formato_valido": metricas_baseline["formato_valido"],
        "baseline/rouge_l": metricas_baseline["rouge_l"],
        "finetuned/exactitud": metricas_finetuned["exactitud"],
        "finetuned/formato_valido": metricas_finetuned["formato_valido"],
        "finetuned/rouge_l": metricas_finetuned["rouge_l"],
        "delta/exactitud": metricas_finetuned["exactitud"] - metricas_baseline["exactitud"],
    })
    wandb.finish()
    print("Registro en W&B completado.")
else:
    print("W&B desactivado; las métricas quedaron en resultados/qwen.json")

## 13 · Discusión

Rellenen esta sección **con sus números**, no con los esperados. Las preguntas
guía son las que un evaluador va a hacer:

**1. ¿Qué mejoró exactamente?**
Comparen la variación de `formato_valido` contra la de `exactitud`. Si el
formato subió mucho (por ejemplo de 20% a 95%) y la exactitud subió poco, la
conclusión honesta es: *el fine-tuning enseñó el formato de respuesta, no a
hacer mejor las cuentas*. Es un resultado legítimo y publicable; presentarlo
como "el modelo aprendió matemáticas" no lo sería.

**2. ¿Hay sobreajuste?**
Miren en qué época tocó fondo `eval_loss`. Si fue en la 2 o 3 de 8, el modelo
agotó lo que podía aprender de 132 ejemplos muy pronto. Eso no se arregla
entrenando más épocas: se arregla con más datos.

**3. ¿Dónde falla todavía?**
Revisen el desglose por categoría. Es esperable que las categorías de varios
pasos (ecuaciones, operaciones combinadas, porcentajes con dos operaciones)
vayan peor que las de un solo paso: cada paso adicional multiplica la
probabilidad de un error aritmético.

**4. ¿Qué significan los ejemplos que empeoraron?**
Si hay ejemplos que el baseline resolvía y el modelo entrenado ya no, vale la
pena mirarlos uno a uno. Suele indicar que el modelo aprendió a imitar la
*forma* de nuestras respuestas cortas y abandonó un razonamiento más largo que
antes le funcionaba.

## 14 · Conclusiones

Cierren con tres puntos concretos:

1. **Resultado cuantitativo.** Exactitud baseline → exactitud fine-tuned, sobre
   33 ejemplos de validación. Mencionen el intervalo: con n=33, una diferencia
   de un solo ejemplo son 3 puntos porcentuales. Diferencias menores a ~10
   puntos no son concluyentes con este tamaño de muestra, y decirlo es parte
   del rigor del experimento.
2. **Qué aprendió el modelo.** Formato, estilo de explicación, longitud
   adecuada, uso del español. Y qué no: aritmética confiable.
3. **Qué haría falta.** Más datos (el corpus está preparado para escalar
   reemplazando el JSONL), o un enfoque híbrido donde el modelo genere el
   procedimiento y una herramienta externa verifique el cálculo.

### Qué sigue

- **Notebook 2 (BERT):** clasificar el tipo de problema con un encoder.
- **Notebook 3 (Qwen+BERT):** usar esa clasificación para especializar el
  prompt de generación y medir si el enrutamiento aporta.
- **Notebook 4 (FLAN-T5):** resolver clasificación y generación en un solo
  modelo encoder-decoder.
- **Notebook 5:** comparación de las cuatro arquitecturas.